Script para definir qual será o método final usado para o restante dos testes para inclusão no artigo. A seleção funcionará da seguinte forma:

- Para *BRKGAs* (puro e com mutação personalizada)
1. comparação entre mesmo método com diferents valores de stag (fixar o melhor valor de stag)
2. comparação entre métodos diferentes (com o parâmetro de stag que melhora a performance de cada um já definido)
3. o método que "vence" no passo 2 será o escolhido para a apresentação de resultados no artigo

- Para *Genéticos* (simple_mean e binomial) 
1. comparação entre mesmo método com mesma probabilidade e diferentes valores de stag (primeiro, fixamos o valor de stag para cada possível probabilidade testada)
2. comparação entre mesmo método com diferentes probabilidades (com o valor de stag já fixado)
    * a ideia é que apoós esse passo tenhamos definido qual é a melhor combinação de parâmetros para cada método, e partir disso compararemos as melhores versões para decidir qual será o método final utilizado
3. comparação entre métodos diferentes (com valores de probabilidade e stag que maximizam cada performance já definidos)
4. o método que "vence" no passo 3 será o escolhido para apresentação de resultados no 

* as análises serão separadas por classes de grafos - primeiro grafos bipartidos e depois grafos simples

**PARTE 1 - TESTES DE PERFORMANCE PARA GRAFOS BIPARTIDOS**

**PARTE 1.1 - TESTES COM BRKGAs**

In [4]:
import pandas as pd
import numpy as np

# --- 1. YOUR HELPER FUNCTION ---
def get_detailed_stats(row):
    # Use .get() to avoid KeyError if columns are missing entirely
    required_cols = ['mean_chi_Stag20', 'mean_chi_Stag50', 'mean_time_Stag20', 'mean_time_Stag50']
    if any(pd.isna(row.get(col)) for col in required_cols):
        return 'Incomplete', 'Incomplete', 'Incomplete'
    
    # QUALITY (Chi)
    if row['mean_chi_Stag20'] < row['mean_chi_Stag50']:
        chi_winner = 'Stag 20'
    elif row['mean_chi_Stag50'] < row['mean_chi_Stag20']:
        chi_winner = 'Stag 50'
    else:
        chi_winner = 'Tied'

    # SPEED (Time)
    if row['mean_time_Stag20'] < row['mean_time_Stag50']:
        time_winner = 'Stag 20'
    elif row['mean_time_Stag50'] < row['mean_time_Stag20']:
        time_winner = 'Stag 50'
    else:
        time_winner = 'Tied'

    # ABSOLUTE WINNER
    abs_winner = chi_winner if chi_winner != 'Tied' else time_winner 
    return chi_winner, time_winner, abs_winner

# --- 2. THE MAIN ANALYSIS FUNCTION ---
def analyze_brkga_performance(files_dict, title_name):
    all_results = []
    cols_to_keep = ['instancia', 'mean_chi', 'mean_time']
    
    for label, file_path in files_dict.items():
        df = pd.read_csv(file_path)
        # Clean column names in case CSV has leading/trailing spaces
        df.columns = df.columns.str.strip()
        
        df_filtered = df[[c for c in cols_to_keep if c in df.columns]].copy()
        df_filtered['parameter_id'] = label
        all_results.append(df_filtered)
    
    df_combined = pd.concat(all_results, ignore_index=True)
    
    # Pivot to side-by-side (Removing spaces from labels so keys match your helper function)
    df_pivot = df_combined.pivot(index='instancia', columns='parameter_id', values=['mean_chi', 'mean_time'])
    df_pivot.columns = [f'{col}_{val}'.replace(' ', '') for col, val in df_pivot.columns]
    df_pivot = df_pivot.reset_index()

    # Apply the Winner Logic
    df_pivot[['chi_winner', 'time_winner', 'abs_winner']] = df_pivot.apply(
        lambda r: pd.Series(get_detailed_stats(r)), axis=1
    )

    valid = df_pivot[df_pivot['chi_winner'] != 'Incomplete']

    # PRINT SUMMARY
    print("\n" + "="*60)
    print(f"{title_name:^60}")
    print("="*60)
    if not valid.empty:
        print(f"STRICTLY BY CHI:\n{valid['chi_winner'].value_counts().to_string()}\n")
        print(f"STRICTLY BY TIME:\n{valid['time_winner'].value_counts().to_string()}\n")
        print(f"ABSOLUTE WINNER:\n{valid['abs_winner'].value_counts().to_string()}")
    else:
        print("No valid comparisons - check if instances match in both files.")
    print("="*60 + "\n")

    return df_pivot

# --- 3. RUNNING THE ANALYSIS ---

# Define your Bi files
pure_files = {
    "Stag 20": "results_BRKGA_Bi_stag20_progresso1.csv",
    "Stag 50": "results_BRKGA_Bi_stag50_progresso1.csv"
}

mutation_files = {
    "Stag 20": "results_BRKGAmutation_Bi_stag20_progresso1.csv",
    "Stag 50": "results_BRKGAmutation_Bi_stag50_progresso1.csv"
}

# Run and store
df_pure_pivot = analyze_brkga_performance(pure_files, "RESULTS: PURE BRKGA (Bi)")
df_mut_pivot = analyze_brkga_performance(mutation_files, "RESULTS: BRKGA WITH MUTATION (Bi)")

# --- 4. QUICK CHECK: STAG 50 MARGINS ---
def check_stag50_margins(df_to_check, analysis_type):
    if df_to_check is None: return
    stag50_wins = df_to_check[df_to_check['chi_winner'] == 'Stag 50'].copy()

    if not stag50_wins.empty:
        stag50_wins['chi_diff'] = (stag50_wins['mean_chi_Stag20'] - stag50_wins['mean_chi_Stag50']).round(4)
        print("\n" + "="*80)
        print(f"{f'STAG 50 QUALITY WINS ({analysis_type}): DETAILED MARGINS':^80}")
        print("="*80)
        cols = ['instancia', 'mean_chi_Stag20', 'mean_chi_Stag50', 'chi_diff']
        print(stag50_wins[cols].sort_values('chi_diff', ascending=False).to_string(index=False))
        print("="*80)
    else:
        print(f"\n[!] No quality wins for Stag 50 in {analysis_type}.")

check_stag50_margins(df_pure_pivot, "PURE")
check_stag50_margins(df_mut_pivot, "MUTATION")


                  RESULTS: PURE BRKGA (Bi)                  
STRICTLY BY CHI:
chi_winner
Stag 50    34
Stag 20    26
Tied       12

STRICTLY BY TIME:
time_winner
Stag 20    72

ABSOLUTE WINNER:
abs_winner
Stag 20    38
Stag 50    34


             RESULTS: BRKGA WITH MUTATION (Bi)              
STRICTLY BY CHI:
chi_winner
Stag 50    53
Stag 20    12
Tied        7

STRICTLY BY TIME:
time_winner
Stag 20    72

ABSOLUTE WINNER:
abs_winner
Stag 50    53
Stag 20    19


                 STAG 50 QUALITY WINS (PURE): DETAILED MARGINS                  
               instancia  mean_chi_Stag20  mean_chi_Stag50  chi_diff
bi_a100_b500_p01%_v1.col             84.8             81.4       3.4
bi_a100_b500_p05%_v2.col             99.4             98.0       1.4
bi_a500_b500_p05%_v1.col            298.6            297.4       1.2
bi_a500_b500_p03%_v2.col            202.8            201.8       1.0
bi_a100_b500_p70%_v2.col            584.8            584.0       0.8
bi_a100_b100_p01%_v2.col          

**CONCLUSION**
For the pure BRKGA, the gain in coloring quality is not considerable enought for stag = 50 to be considered the better parameter value. We will use pure BRKGA with stag = 20 for bipartite graphs.

For the BRKGAmutation, the version with stag = 50 performs better both in terms of coloring, but the gain (the biggest one is of 4 colors only, not even 5% improvement) is not as considerable if the time overhead is taken into account. Therefore, we choose BRKGAmutation with stag = 20 as well.

In [13]:
# 2B. comparison betwwen the different methods with already fixed stag values to check which one is beter
# 3B. tomada de decisão sobre qual é o melhor dos BRKGA's (grafos simples)
import pandas as pd

# logic for comparing methods with stag value already determined
def get_method_stats(row):
    # labels are now pure and mutation
    required_cols = ['mean_chi_Pure', 'mean_chi_Mutation', 'mean_time_Pure', 'mean_time_Mutation']
    if any(pd.isna(row.get(col)) for col in required_cols):
        return 'Incomplete', 'Incomplete', 'Incomplete'
    
    if row['mean_chi_Mutation'] < row['mean_chi_Pure']:
        chi_winner = 'Mutation'
    elif row['mean_chi_Pure'] < row['mean_chi_Mutation']:
        chi_winner = 'Pure'
    else:
        chi_winner = 'Tied'

    if row['mean_time_Mutation'] < row['mean_time_Pure']:
        time_winner = 'Mutation'
    elif row['mean_time_Pure'] < row['mean_time_Mutation']:
        time_winner = 'Pure'
    else:
        time_winner = 'Tied'

    abs_winner = chi_winner if chi_winner != 'Tied' else time_winner
    
    return chi_winner, time_winner, abs_winner

# comparison files (locked at stag 20)
comparison_files = {
    "Pure": "results_BRKGA_Bi_stag20_progresso1.csv",
    "Mutation": "results_BRKGAmutation_Bi_stag20_progresso1.csv"
}

# load and processing
all_results = []
for label, path in comparison_files.items():
    df = pd.read_csv(path)
    df_f = df[['instancia', 'mean_chi', 'mean_time']].copy()
    df_f['method_id'] = label
    all_results.append(df_f)

df_comp = pd.concat(all_results, ignore_index=True)
df_pivot = df_comp.pivot(index='instancia', columns='method_id', values=['mean_chi', 'mean_time'])
df_pivot.columns = [f'{col}_{val}'.replace(' ', '') for col, val in df_pivot.columns]
df_pivot = df_pivot.reset_index()

# check winners
df_pivot[['chi_winner', 'time_winner', 'abs_winner']] = df_pivot.apply(
    lambda r: pd.Series(get_method_stats(r)), axis=1
)

# result printing
valid = df_pivot[df_pivot['chi_winner'] != 'Incomplete']

print("\n" + "="*60)
print(f"{'METHOD COMPARISON: PURE vs MUTATION (FIXED STAG=20)':^60}")
print("="*60)
print(f"STRICTLY BY CHI (Quality):\n{valid['chi_winner'].value_counts().to_string()}\n")
print(f"STRICTLY BY TIME (Speed):\n{valid['time_winner'].value_counts().to_string()}\n")
print(f"ABSOLUTE WINNER (Chi > Time):\n{valid['abs_winner'].value_counts().to_string()}")
print("="*60 + "\n")

mut_quality_wins = valid[valid['chi_winner'] == 'Mutation'].copy()

if not mut_quality_wins.empty:
    # 2. Calculate Quality Improvement
    mut_quality_wins['chi_diff'] = (
        mut_quality_wins['mean_chi_Pure'] - mut_quality_wins['mean_chi_Mutation']
    ).round(4)

    # 3. Calculate Time Penalty (How much longer it took)
    mut_quality_wins['extra_time_sec'] = (
        mut_quality_wins['mean_time_Mutation'] - mut_quality_wins['mean_time_Pure']
    ).round(4)
    
    # Calculate % increase in time
    mut_quality_wins['time_increase_pct'] = (
        (mut_quality_wins['extra_time_sec'] / mut_quality_wins['mean_time_Pure']) * 100
    ).round(2)

    # Sort by the biggest quality improvement
    mut_quality_wins = mut_quality_wins.sort_values(by='chi_diff', ascending=False)

    print("\n" + "="*95)
    print(f"{'MUTATION QUALITY WINS: QUALITY GAIN vs TIME COST':^95}")
    print("="*95)
    
    # Display the metrics
    cols = ['instancia', 'chi_diff', 'mean_time_Pure', 'mean_time_Mutation', 'extra_time_sec', 'time_increase_pct']
    # Renaming for cleaner display
    display_df = mut_quality_wins[cols].rename(columns={
        'chi_diff': 'Chi Gain',
        'extra_time_sec': '+Time (s)',
        'time_increase_pct': '+Time (%)'
    })
    
    print(display_df.to_string(index=False))
    
    print("-" * 95)
    avg_extra = mut_quality_wins['extra_time_sec'].mean()
    avg_pct = mut_quality_wins['time_increase_pct'].mean()
    print(f"On average, Mutation found better colors but took {avg_extra:.2f}s longer ({avg_pct:.2f}% increase).")
    print("="*95)
else:
    print("\n[!] No quality wins found for Mutation vs Pure at Stag 20.")


    METHOD COMPARISON: PURE vs MUTATION (FIXED STAG=20)     
STRICTLY BY CHI (Quality):
chi_winner
Mutation    58
Tied        10
Pure         4

STRICTLY BY TIME (Speed):
time_winner
Pure        58
Mutation    14

ABSOLUTE WINNER (Chi > Time):
abs_winner
Mutation    62
Pure        10


                       MUTATION QUALITY WINS: QUALITY GAIN vs TIME COST                        
               instancia  Chi Gain  mean_time_Pure  mean_time_Mutation  +Time (s)  +Time (%)
bi_a500_b500_p01%_v1.col      85.8       14.932027           52.853793    37.9218     253.96
bi_a500_b500_p01%_v2.col      83.8       15.111490           50.783448    35.6720     236.06
bi_a100_b500_p01%_v1.col      22.6        1.946286            4.513415     2.5671     131.90
bi_a100_b500_p01%_v2.col      16.4        2.021375            4.510559     2.4892     123.14
bi_a500_b500_p10%_v2.col      16.0       43.093518          221.448536   178.3550     413.88
bi_a100_b100_p03%_v1.col      12.2        0.545904        

**CONCLUSION**
The BRKGAmutation (with stag = 20) gets better coloring results in most cases (with an improvement of up to 85.8 colors) with a not so big time overhead. Therefore, the chosen method is BRKGAmutation with stag = 20 for bipartite graphs. 